# DT CultureTech — Business Analytics Target Company Research
## Analysis Notebook | Part A Submission

> **Purpose:** Load the target company dataset, compute scores, generate visualizations, and summarize insights.
> 
> **Author:** Business Analytics Intern Candidate
> **Date:** May 2026

---

### How to Use This Notebook
1. Run all cells in order (Cell → Run All)
2. Charts will be saved to `../charts/` folder automatically
3. All scoring logic is visible and auditable below
4. Modify `ICP_WEIGHTS` in Cell 3 to experiment with different scoring priorities

## Cell 1 — Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
plt.style.use('dark_background')

print('✅ Libraries loaded successfully')
print(f'   Pandas version: {pd.__version__}')

## Cell 2 — Load and Inspect the Dataset

In [ ]:
# Load the target company dataset
CSV_PATH = '../target_company_analysis.csv'

df = pd.read_csv(CSV_PATH)

print(f'Dataset loaded: {df.shape[0]} companies, {df.shape[1]} columns')
print('\nColumn list:')
for col in df.columns:
    print(f'  • {col}')

print('\nFirst 3 rows preview:')
df[['Company Name', 'Industry', 'Total Score', 'Final Verdict']].head(3)

## Cell 3 — Scoring Framework & Configuration

> **Analyst Note:** The 6 criteria weights below are currently equal (each out of 5, total out of 30).
> You can modify `ICP_WEIGHTS` to give more weight to criteria you consider more important.
> For example, if Domain Alignment (C6) is most critical, set its weight to 2.0.

In [ ]:
# Scoring criteria columns
SCORE_COLS = [
    'C1_Innovation_Culture',
    'C2_Team_Size_Fit',
    'C3_DM_LinkedIn_Access',
    'C4_Budget_Ready',
    'C5_Personalization_Hook',
    'C6_Domain_Alignment'
]

# Equal weights (default) — modify here to experiment
ICP_WEIGHTS = {
    'C1_Innovation_Culture':    1.0,   # Innovation culture readiness
    'C2_Team_Size_Fit':         1.0,   # Team size in DT's sweet spot
    'C3_DM_LinkedIn_Access':    1.0,   # Decision maker reachable via LinkedIn
    'C4_Budget_Ready':          1.0,   # Funding/revenue signals budget availability
    'C5_Personalization_Hook':  1.0,   # Strong specific outreach trigger exists
    'C6_Domain_Alignment':      1.0,   # How naturally DT's pitch fits the sector
}

# Verdict thresholds
VERDICT_THRESHOLDS = {'HOT': 22, 'WARM': 14}

# Verify total scores match
df['Computed_Total'] = df[SCORE_COLS].sum(axis=1)
score_match = (df['Total Score'] == df['Computed_Total']).all()
print(f'Score verification: {"✅ All totals match" if score_match else "❌ Mismatch found"}')

# Score distribution summary
print('\nScore Statistics:')
print(df['Total Score'].describe().round(2))

## Cell 4 — Verdict Distribution & Pipeline Summary

In [ ]:
# Pipeline summary by verdict
verdict_counts = df['Final Verdict'].value_counts()

print('='*50)
print('DT CULTURETECH — OUTREACH PIPELINE SUMMARY')
print('='*50)
for verdict, count in verdict_counts.items():
    pct = count / len(df) * 100
    bar = '█' * count
    print(f'{verdict:<6} {bar:<20} {count:>2} companies ({pct:.0f}%)')

print(f'\nTotal Pipeline:  {len(df)} companies')
print(f'Average Score:   {df["Total Score"].mean():.1f} / 30')
print(f'Highest Score:   {df["Total Score"].max()} ({df.loc[df["Total Score"].idxmax(), "Company Name"]})')
print(f'Lowest Score:    {df["Total Score"].min()} ({df.loc[df["Total Score"].idxmin(), "Company Name"]})')

print('\nTop 10 Targets by Score:')
top10 = df.nlargest(10, 'Total Score')[['Company Name', 'Industry', 'Total Score', 'Final Verdict']]
top10.index = range(1, 11)
print(top10.to_string())

## Cell 5 — Criteria Average Analysis

> **Analyst Insight:** Which criteria scored highest/lowest across all companies?
> Criteria with consistently low scores indicate gaps in the pipeline — for example, if C4 (Budget Ready) is low on average, the pipeline has too many under-funded companies.

In [ ]:
criteria_labels = {
    'C1_Innovation_Culture':   'C1: Innovation Culture',
    'C2_Team_Size_Fit':        'C2: Team Size Fit',
    'C3_DM_LinkedIn_Access':   'C3: DM LinkedIn Access',
    'C4_Budget_Ready':         'C4: Budget Ready',
    'C5_Personalization_Hook': 'C5: Personalization Hook',
    'C6_Domain_Alignment':     'C6: Domain Alignment'
}

criteria_means = df[SCORE_COLS].mean()
criteria_means.index = [criteria_labels[c] for c in SCORE_COLS]

print('Average Score by Criterion (out of 5):')
print('─' * 45)
for criterion, mean in criteria_means.sort_values(ascending=False).items():
    bar = '█' * int(mean * 4)
    print(f'{criterion:<28} {mean:.2f}  {bar}')

print(f'\n→ Highest average: {criteria_means.idxmax()} ({criteria_means.max():.2f})')
print(f'→ Lowest average:  {criteria_means.idxmin()} ({criteria_means.min():.2f})')

# Insight
print('\nKey Insight:')
if criteria_means.idxmin() == 'C6: Domain Alignment':
    print('  Pipeline skews toward non-EdTech sectors — consider adding more EdTech targets')
elif criteria_means.idxmin() == 'C4: Budget Ready':
    print('  Many targets are under-funded — refine ICP to include more Series B+ companies')
else:
    print(f'  Weakest dimension is {criteria_means.idxmin()} — review scoring criteria for this dimension')

## Cell 6 — Industry Distribution Analysis

In [ ]:
# Simplify industry labels
def simplify_industry(ind):
    if 'EdTech' in str(ind):     return 'EdTech'
    elif 'FinTech' in str(ind):  return 'FinTech'
    elif 'HRTech' in str(ind):   return 'HRTech'
    elif 'AgriTech' in str(ind): return 'AgriTech'
    elif 'InsurTech' in str(ind):return 'InsurTech'
    elif 'HealthTech' in str(ind):return 'HealthTech'
    elif 'SaaS' in str(ind):     return 'SaaS / CRM'
    elif 'Retail' in str(ind):   return 'RetailTech'
    elif 'LoyaltyTech' in str(ind): return 'LoyaltyTech'
    return ind

df['Industry_Simple'] = df['Industry'].apply(simplify_industry)
industry_counts = df['Industry_Simple'].value_counts()

print('Industry Distribution:')
print(industry_counts.to_string())

# Average score by industry
print('\nAverage Score by Industry:')
industry_scores = df.groupby('Industry_Simple')['Total Score'].mean().sort_values(ascending=False)
print(industry_scores.round(1).to_string())

## Cell 7 — Correlation Analysis (Which Criteria Predict Total Score Best?)

In [ ]:
# Correlation between each criterion and total score
correlations = df[SCORE_COLS + ['Total Score']].corr()['Total Score'].drop('Total Score')
correlations.index = [criteria_labels[c] for c in SCORE_COLS]

print('Correlation with Total Score (higher = more influential):')
print('─' * 50)
for criterion, corr in correlations.sort_values(ascending=False).items():
    bar = '█' * int(corr * 20)
    print(f'{criterion:<28} {corr:.3f}  {bar}')

print(f'\n→ Most predictive criterion: {correlations.idxmax()}')
print('→ Analyst Insight: The criterion with highest correlation is where')
print('  scoring differences between HOT and WARM companies are most concentrated.')

## Cell 8 — Outreach Priority List (Final Ranked Output)

In [ ]:
# Final ranked outreach list
output_cols = [
    'Company Name', 'Industry_Simple', 'Decision Maker Name', 'Role',
    'Total Score', 'Final Verdict', 'Personalization Hook'
]

ranked = df.sort_values('Total Score', ascending=False)[output_cols].reset_index(drop=True)
ranked.index += 1  # Start ranking from 1

print('RANKED OUTREACH PRIORITY LIST')
print('='*80)
for idx, row in ranked.iterrows():
    verdict_icon = '🔴' if row['Final Verdict'] == 'HOT' else '🟡'
    print(f"{idx:>2}. {verdict_icon} [{row['Total Score']}/30] {row['Company Name']:<25} | {row['Decision Maker Name']}")
    if idx <= 5:  # Show hooks for top 5
        hook_preview = str(row['Personalization Hook'])[:80] + '...'
        print(f'    Hook: {hook_preview}')

## Cell 9 — Save All Charts

> This cell regenerates all three charts and saves them to the `../charts/` folder.

In [ ]:
os.makedirs('../charts', exist_ok=True)
verdict_colors = {'HOT': '#f85149', 'WARM': '#d29922', 'COLD': '#8b949e'}

# ─── CHART 1: Industry Distribution ───────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

colors_palette = ['#58a6ff','#3fb950','#d29922','#f85149','#bc8cff','#56d364','#e8b55a','#39d353']
bars = ax.barh(industry_counts.index, industry_counts.values,
               color=colors_palette[:len(industry_counts)], edgecolor='#30363d')
for bar, val in zip(bars, industry_counts.values):
    ax.text(val+0.05, bar.get_y()+bar.get_height()/2, f'  {val}', va='center', color='#c9d1d9', fontsize=10, fontweight='bold')

ax.set_title('Industry Distribution — 25 Target Companies\n(DT CultureTech B2B Pipeline)', color='#f0f6fc', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Companies', color='#8b949e')
for spine in ['top','right']: ax.spines[spine].set_visible(False)
for spine in ['bottom','left']: ax.spines[spine].set_color('#30363d')
ax.tick_params(colors='#c9d1d9')
plt.tight_layout()
plt.savefig('../charts/industry_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('✅ Chart 1 saved')

# ─── CHART 2: Score Distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0d1117')

ax1, ax2 = axes
ax1.set_facecolor('#161b22'); ax2.set_facecolor('#161b22')

for v, c in verdict_colors.items():
    subset = df[df['Final Verdict']==v]['Total Score']
    if len(subset): ax1.hist(subset, bins=range(10,31), alpha=0.7, color=c, label=f'{v} ({len(subset)})', edgecolor='#0d1117')

ax1.axvline(x=22, color='#f85149', linestyle='--', alpha=0.7, label='HOT (22)')
ax1.axvline(x=14, color='#d29922', linestyle='--', alpha=0.7, label='WARM (14)')
ax1.set_title('Score Distribution by Verdict', color='#f0f6fc', fontsize=11, fontweight='bold')
ax1.set_xlabel('Total Score (out of 30)', color='#8b949e')
ax1.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=8)
for spine in ['top','right']: ax1.spines[spine].set_visible(False)
ax1.tick_params(colors='#c9d1d9')

criteria_avgs = df[SCORE_COLS].mean()
short_labels = ['C1\nInnovation','C2\nTeam Size','C3\nLinkedIn','C4\nBudget','C5\nHook','C6\nDomain']
ax2.bar(short_labels, criteria_avgs.values, color=colors_palette[:6], edgecolor='#30363d')
for i, v in enumerate(criteria_avgs.values):
    ax2.text(i, v+0.05, f'{v:.1f}', ha='center', color='#c9d1d9', fontsize=9, fontweight='bold')
ax2.set_ylim(0, 5.5)
ax2.set_title('Average Score per Criterion', color='#f0f6fc', fontsize=11, fontweight='bold')
ax2.set_ylabel('Average (out of 5)', color='#8b949e')
for spine in ['top','right']: ax2.spines[spine].set_visible(False)
ax2.tick_params(colors='#c9d1d9', labelsize=8)

plt.suptitle('DT CultureTech — Pipeline Score Analysis', color='#f0f6fc', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../charts/score_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('✅ Chart 2 saved')

# ─── CHART 3: Top 10 Companies ─────────────────────────────────────────
top10 = df.nlargest(10, 'Total Score').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

bar_colors = [verdict_colors[v] for v in top10['Final Verdict']]
bars = ax.barh(top10['Company Name'][::-1], top10['Total Score'][::-1],
               color=bar_colors[::-1], edgecolor='#30363d')
for bar, val in zip(bars, top10['Total Score'][::-1]):
    ax.text(val+0.1, bar.get_y()+bar.get_height()/2, f'{val}/30', va='center', color='#c9d1d9', fontsize=10, fontweight='bold')

ax.set_xlim(0, 32)
ax.set_title('Top 10 Target Companies by Score\n(DT CultureTech Priority Pipeline)', color='#f0f6fc', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Score (out of 30)', color='#8b949e')
patches = [mpatches.Patch(color=c, label=v) for v, c in verdict_colors.items() if v != 'COLD']
ax.legend(handles=patches, facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9')
for spine in ['top','right']: ax.spines[spine].set_visible(False)
ax.tick_params(colors='#c9d1d9')
plt.tight_layout()
plt.savefig('../charts/employee_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('✅ Chart 3 saved')
print('\n✅ All charts regenerated and saved to ../charts/')

## Cell 10 — Final Summary & Analyst Takeaways

In [ ]:
print('='*60)
print('FINAL ANALYST SUMMARY')
print('='*60)

hot = df[df['Final Verdict']=='HOT']
warm = df[df['Final Verdict']=='WARM']

print(f'\n📊 Pipeline Quality:')
print(f'   Total Companies:    {len(df)}')
print(f'   HOT (≥22):          {len(hot)} ({len(hot)/len(df)*100:.0f}%)')
print(f'   WARM (14–21):       {len(warm)} ({len(warm)/len(df)*100:.0f}%)')
print(f'   Average Score:      {df["Total Score"].mean():.1f}/30')

print(f'\n🎯 Top 5 Immediate Outreach Targets:')
for i, (_, row) in enumerate(df.nlargest(5,'Total Score').iterrows(), 1):
    print(f'   {i}. {row["Company Name"]} ({row["Total Score"]}/30) — {row["Decision Maker Name"]}')

print(f'\n💡 Key Insights:')
top_industry = df.groupby('Industry_Simple')['Total Score'].mean().idxmax()
print(f'   • {top_industry} companies score highest on average — DT\'s primary vertical')
print(f'   • Companies with LinkedIn Presence Score 5/5: {(df["LinkedIn Presence Score"]==5).sum()} companies')
print(f'   • Existing DT clients included: Growpital, Kringle.ai, IPx (account expansion track)')
print(f'   • 3 Hyderabad-based HOT targets (local relationship advantage for DT)')

print(f'\n⚠️  Research Limitations:')
print('   • Revenue figures are estimated bands, not audited actuals')
print('   • LinkedIn data is point-in-time — re-verify before outreach')
print('   • Growpital carries SEBI regulatory risk — monitor for resolution')
print('   • Personalization hooks have a 6-month relevance shelf life')

print('\n' + '='*60)
print('END OF ANALYSIS NOTEBOOK')
print('='*60)